In [1]:
import tensorflow as tf
import keras
from keras import layers

import matplotlib.pyplot as plt
import numpy as np
import random

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
filepath = '/content/drive/MyDrive/hobbit.txt'
with open(filepath) as f:
    text = f.read()

print("Corpus length:", len(text))

text = text.lower()
text_vec_layer = tf.keras.layers.TextVectorization(split = "character", standardize = "lower")
text_vec_layer.adapt([text])

n_tokens = text_vec_layer.vocabulary_size() #0 = padding, 1 = unknown
print("Total chars:", n_tokens)

maxlen = 40
step = 3
sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i:i + maxlen])
    next_chars.append(text[i + maxlen])
print("Number of sequences:", len(sentences))

x = np.asarray(text_vec_layer(sentences))
y = np.asarray(text_vec_layer(next_chars))

x_train, x_valid, y_train, y_valid = train_test_split(x, y, random_state = 42, shuffle = True)

Corpus length: 505897
Total chars: 51
Number of sequences: 168619


In [7]:
batch_size = 128

embed = keras.Sequential([
    layers.Embedding(input_dim = n_tokens, output_dim = 16)
])

rnn = keras.Sequential([
    layers.LSTM(128),
    layers.Dense(n_tokens, activation = "softmax")
])

model = keras.Sequential([embed, rnn])

optimizer = keras.optimizers.Nadam(learning_rate = 0.01)
model.compile(loss = "sparse_categorical_crossentropy", optimizer = optimizer, metrics = ["accuracy"])

epochs = 40
lr = keras.callbacks.ReduceLROnPlateau(monitor = "val_loss", factor = 0.5, patience = 3)
es = keras.callbacks.EarlyStopping(monitor = "val_loss", patience = 5, restore_best_weights = True)
model.fit(x_train, y_train, batch_size = batch_size, epochs = epochs, validation_data = (x_valid, y_valid), callbacks = [es, lr])


Epoch 1/40
988/988 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.3641 - loss: 2.2167 - val_accuracy: 0.4978 - val_loss: 1.6767 - learning_rate: 0.0100
Epoch 2/40
988/988 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.5130 - loss: 1.6253 - val_accuracy: 0.5269 - val_loss: 1.5665 - learning_rate: 0.0100
Epoch 3/40
988/988 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.5407 - loss: 1.5093 - val_accuracy: 0.5443 - val_loss: 1.5158 - learning_rate: 0.0100
Epoch 4/40
988/988 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.5565 - loss: 1.4514 - val_accuracy: 0.5537 - val_loss: 1.4989 - learning_rate: 0.0100
Epoch 5/40
988/988 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.5636 - loss: 1.4236 - val_accuracy: 0.5515 - val_loss: 1.4921 - learning_rate: 0.0100
Epoch 6/40
988/988 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.5720 - loss: 1.3950 - val_accuracy: 0.5580 - val_loss: 1.4696 - learning_rate: 0.0100
Epoch 7/40
988/988 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.5740 - loss: 1.3786 -

In [8]:
tolkien_model = tf.keras.Sequential([
    text_vec_layer,
    model
])

In [9]:
def str_array_fix(a):
    return(np.array(a).astype(object))

In [10]:
def pad_sentence(sentence):
    if len(sentence) < maxlen:
        sentence = " "*(maxlen-len(sentence)) + sentence
    elif len(sentence) > maxlen:
        sentence = sentence[:-maxlen]
    return sentence

In [11]:
sentence = "gollu"
y_proba = tolkien_model.predict(str_array_fix([pad_sentence(sentence)]), verbose = False)[0]
y_pred = tf.argmax(y_proba)
text_vec_layer.get_vocabulary()[y_pred]

np.str_('m')

In [12]:
def next_char(sentence, temperature=1):
    y_proba = tolkien_model.predict(str_array_fix([pad_sentence(sentence)]), verbose = False)[0]
    rescaled_logits = tf.math.log([y_proba]) / temperature
    char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0]
    return text_vec_layer.get_vocabulary()[char_id]

In [13]:
def extend_text(sentence, n_chars = 50, temperature = 1):
    for _ in range(n_chars):
        sentence += next_char(sentence, temperature)
    return sentence

In [18]:
np.random.seed(32)

sentence = "gand"
print(extend_text(sentence, n_chars = 100, temperature = 0.01))

gandalf were standing days the dwarves anhidin  ware standing tors the dwarves andtsesgwahsm aaanding to
